## Module A — Data preprocessing & merge (**daily**, corrected)\n\nThis notebook replaces the earlier (incorrect) annual-mean pipeline. It:\n\n- Cleans **NFLX** and **GLD** daily data\n- Aligns the shared trading calendar via an **inner join on `Date`**\n- Quantifies coverage / missingness\n- Exports daily aligned **prices** and **log returns** for downstream modules\n\nOutputs:\n- `outputs/combined_daily_prices.csv`\n- `outputs/combined_daily_logreturns.csv`\n

In [ ]:
from pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nROOT = Path.cwd()\nOUT_DIR = ROOT / "outputs"\nOUT_DIR.mkdir(exist_ok=True)\n\nnflx_path = ROOT / "NFLX.csv"\ngld_path = ROOT / "navhist-us-en-gld(navhist).csv"\n\nnflx_path, gld_path\n

In [ ]:
# --- Load & clean NFLX ---\nnflx = pd.read_csv(nflx_path)\nnflx["Date"] = pd.to_datetime(nflx["Date"], errors="coerce")\nnflx["Adj Close"] = pd.to_numeric(nflx["Adj Close"], errors="coerce")\nnflx = nflx.dropna(subset=["Date", "Adj Close"]).sort_values("Date")\nnflx = nflx[["Date", "Adj Close"]].rename(columns={"Adj Close": "NFLX_AdjClose"})\n\nnflx.head()\n

In [ ]:
# --- Load & clean GLD (metadata rows + encoding fallback) ---\nlast_err = None\nfor enc in ("utf-8-sig", "cp1252", "latin1"):\n    try:\n        gld = pd.read_csv(gld_path, skiprows=3, encoding=enc)\n        last_err = None\n        break\n    except Exception as e:\n        last_err = e\nif last_err is not None:\n    raise last_err\n\ngld["Date"] = pd.to_datetime(gld["Date"], format="%d-%b-%Y", errors="coerce")\ngld["NAV"] = pd.to_numeric(gld["NAV"], errors="coerce")\ngld = gld.dropna(subset=["Date", "NAV"]).sort_values("Date")\ngld = gld[["Date", "NAV"]].rename(columns={"NAV": "GLD_NAV"})\n\ngld.head()\n

In [ ]:
# --- Align trading days and quantify coverage ---\nmerged_prices = (\n    pd.merge(nflx, gld, on="Date", how="inner")\n    .sort_values("Date")\n    .reset_index(drop=True)\n)\n\ncoverage = {\n    "nflx_days": int(len(nflx)),\n    "gld_days": int(len(gld)),\n    "merged_days": int(len(merged_prices)),\n    "min_date": str(merged_prices["Date"].min().date()),\n    "max_date": str(merged_prices["Date"].max().date()),\n}\ncoverage\n

In [ ]:
# --- Compute daily log returns ---\nmerged = merged_prices.copy()\nmerged["NFLX_logret"] = np.log(merged["NFLX_AdjClose"]).diff()\nmerged["GLD_logret"] = np.log(merged["GLD_NAV"]).diff()\nmerged = merged.dropna(subset=["NFLX_logret", "GLD_logret"]).reset_index(drop=True)\n\nmerged.head()\n

In [ ]:
# --- Export for downstream modules ---\nprices_out = OUT_DIR / "combined_daily_prices.csv"\nrets_out = OUT_DIR / "combined_daily_logreturns.csv"\n\nmerged_prices.to_csv(prices_out, index=False)\nmerged.to_csv(rets_out, index=False)\n\nprices_out, rets_out\n